In [3]:
import pandas as pd
import xarray as xr
import numpy as np
import os
import re
import subprocess
from pathlib import Path
import boto3
from botocore.client import Config
from botocore import UNSIGNED
from botocore.config import Config as BotocoreConfig
import fsspec
import s3fs
from io import StringIO

In [ ]:
# Create an S3 client with anonymous access
'''
s3 = boto3.client(
    's3',
    endpoint_url = 'https://sdsc.osn.xsede.org',
    config = Config(
        signature_version = UNSIGNED,
        s3 = {'addressing_style': 'path'}
    )
)
'''

In [4]:
fs = fsspec.filesystem(
    "s3",
    anon = True,
    client_kwargs = {"endpoint_url": "https://sdsc.osn.xsede.org"}
    )

bucket_name = "agr230002-bucket01"
csv_prefix = "hake_data/label_allocations/2017/transect_subgroup_class_dataframes/"
zarr_prefix = "hake_data/data_zarr/MVBS"

In [5]:
# Load region labels from CSV files in S3

def load_region_labels_s3(prefix, bucket = bucket_name):
    # List CSV objects
    all_regions = []
    full_prefix = f"{bucket}/{prefix}"
    
    all_keys = fs.find(full_prefix)
    csv_keys = [k for k in all_keys if k.endswith(".csv")]
    print(f"Found {len(csv_keys)} CSV files in: {full_prefix}")

    for path in csv_keys:
        with fs.open(path, mode="r") as f:
            df = pd.read_csv(f)

            if not df.empty:
                file_name = path.split("/")[-1]
                df["source_file"] = file_name.replace("--all_class_info.csv", "")
                all_regions.append(df)

    if not all_regions:
        return pd.DataFrame()

    combined_df = pd.concat(all_regions, ignore_index=True)

    if "time" not in combined_df.columns:
        return pd.DataFrame()

    results = []
    for _, row in combined_df.iterrows():
        time_list = re.findall(r"'([^']+)'", row["time"])
        if time_list:
            time = pd.to_datetime(time_list)
            time_ms = time.astype("int64") / 10**6
            results.append({
                "id": row["source_file"],
                "start_time": time.min(),
                "end_time": time.max(),
                "start_time_ms": time_ms.min(),
                "end_time_ms": time_ms.max()
            })

    return pd.DataFrame(results).sort_values(by="start_time").reset_index(drop=True)

region_time_df = load_region_labels_s3(prefix = csv_prefix)
region_time_df


Found 176 CSV files in: agr230002-bucket01/hake_data/label_allocations/2017/transect_subgroup_class_dataframes/


,id,start_time,end_time,start_time_ms,end_time_ms
0,x0003_0_wt_20170626_125619_f0013,2017-06-26 13:01:22.781000,2017-06-26 13:27:43.071000,1.498482e+12,1.498484e+12
1,x0003_0_wt_20170626_125619_f0013,2017-06-26 13:58:29.580000,2017-06-26 14:24:16.405000,1.498486e+12,1.498487e+12
2,x0004_0_wt_20170626_191344_f0006,2017-06-26 20:21:13.749000,2017-06-26 20:42:11.118500,1.498508e+12,1.498510e+12
3,x0004_0_wt_20170626_191344_f0006,2017-06-26 20:42:11.118500,2017-06-26 21:23:28.006500,1.498510e+12,1.498512e+12
4,x0004_2_wt_20170627_004507_f0007,2017-06-27 00:28:51.374500,2017-06-27 00:45:13.311000,1.498523e+12,1.498524e+12
...,...,...,...,...,...
266,x0131_0_wt_20170904_141338_f0004,2017-09-04 15:02:31.003500,2017-09-04 15:02:45.599000,1.504537e+12,1.504537e+12
267,x0131_0_wt_20170904_141338_f0004,2017-09-04 15:04:50.926000,2017-09-04 15:05:05.567000,1.504537e+12,1.504538e+12
268,x0131_0_wt_20170904_141338_f0004,2017-09-04 15:05:36.848500,2017-09-04 15:05:49.235000,1.504538e+12,1.504538e+12
269,x0131_2_wt_20170904_172930_f0010,2017-09-04 17:51:59.891000,2017-09-04 18:40:14.303000,1.504548e+12,1.504550e+12


In [ ]:
# Local version of the function

'''
def load_region_labels(csv_dir):
    all_regions = []
    
    # Read all CSV files at once
    for file in os.listdir(csv_dir):
        if file.endswith('.csv'):
            path = os.path.join(csv_dir, file)
            df = pd.read_csv(path)
            if not df.empty:
                df['source_file'] = file.replace('--all_class_info.csv', '')
                all_regions.append(df)
    
    # If no CSV files were found, return an empty DataFrame
    if not all_regions:
        return pd.DataFrame()
    
    # Concatenate all DataFrames into one
    combined_df = pd.concat(all_regions, ignore_index=True)
    
    if 'time' not in combined_df.columns:
        return pd.DataFrame()
    
    # Extract time information from the 'time' column
    results = []
    for _, row in combined_df.iterrows():
        time_list = re.findall(r"'([^']+)'", row['time'])
        if time_list:
            time = pd.to_datetime(time_list)
            time_ms = time.astype('int64') / 10**6
            results.append({
                'id': row['source_file'],
                'start_time': time.min(),
                'end_time': time.max(),
                'start_time_ms': time_ms.min(),
                'end_time_ms': time_ms.max()
            })
    
    return pd.DataFrame(results).sort_values(by=['start_time']).reset_index(drop=True)


csv_dir = 'region_csv'
region_time_df_local = load_region_labels(csv_dir)
region_time_df_local
'''


,id,start_time,end_time,start_time_ms,end_time_ms
0,x0003_0_wt_20170626_125619_f0013,2017-06-26 13:01:22.781000,2017-06-26 13:27:43.071000,1.498482e+12,1.498484e+12
1,x0003_0_wt_20170626_125619_f0013,2017-06-26 13:58:29.580000,2017-06-26 14:24:16.405000,1.498486e+12,1.498487e+12
2,x0004_0_wt_20170626_191344_f0006,2017-06-26 20:21:13.749000,2017-06-26 20:42:11.118500,1.498508e+12,1.498510e+12
3,x0004_0_wt_20170626_191344_f0006,2017-06-26 20:42:11.118500,2017-06-26 21:23:28.006500,1.498510e+12,1.498512e+12
4,x0004_2_wt_20170627_004507_f0007,2017-06-27 00:28:51.374500,2017-06-27 00:45:13.311000,1.498523e+12,1.498524e+12
...,...,...,...,...,...
266,x0131_0_wt_20170904_141338_f0004,2017-09-04 15:02:31.003500,2017-09-04 15:02:45.599000,1.504537e+12,1.504537e+12
267,x0131_0_wt_20170904_141338_f0004,2017-09-04 15:04:50.926000,2017-09-04 15:05:05.567000,1.504537e+12,1.504538e+12
268,x0131_0_wt_20170904_141338_f0004,2017-09-04 15:05:36.848500,2017-09-04 15:05:49.235000,1.504538e+12,1.504538e+12
269,x0131_2_wt_20170904_172930_f0010,2017-09-04 17:51:59.891000,2017-09-04 18:40:14.303000,1.504548e+12,1.504550e+12


In [ ]:
# Load all MVBS zarr files from S3 and extract ping_time range

def load_mvbs_zarr_s3(base_prefix, years, bucket = bucket_name, max_seconds = 1e8): # How to determine 'max_seconds'?
    ping_records = []

    # Loop over each Zarr within that year
    for year in years:
        prefix = f"{base_prefix}/{year}/"
        print(f"Processing year: {year}")

        try:
            all_keys = fs.ls(f"{bucket}/{prefix}", detail=False)
        except Exception as e:
            print(f"Failed to list {prefix}: {e}")
            continue
        
        # List all .zarr folders under the prefix
        zarr_paths = [key for key in all_keys if key.endswith(".zarr")]
        print(f"Found {len(zarr_paths)} zarr in: {prefix}")

        # Open each .zarr and extract ping_time range
        for zarr_key in sorted(zarr_paths):
            try:
                ds = xr.open_zarr(fs.get_mapper(f"s3://{zarr_key}"))
                arr = ds['ping_time'].values

                if np.issubdtype(arr.dtype, np.datetime64):
                    times = pd.to_datetime(arr)
                else:
                    raise ValueError("Not datetime64")

            # Fallback: raw offsets (seconds) + manual conversion
            except ValueError:
                ds = xr.open_zarr(fs.get_mapper(f"s3://{zarr_key}"), decode_times=False)
                raw = ds['ping_time'].values

                # filter out extreme values
                clean = raw[(raw >= 0) & (raw <= max_seconds)]
                if clean.size == 0:
                    print(f"Skipping {zarr_key}: no valid ping times")
                    continue
                    
                print(len(raw) - len(clean), "invalid ping times in:", zarr_key)

                # parse “<unit> since <ref>”
                units = ds["ping_time"].attrs.get("units", "")
                m = re.search(r"(\w+)\s+since\s+(.+)", units)
                if not m:
                    print(f"Skipping {zarr_key}: malformed units “{units}”")
                    continue
                
                # map to pandas timedelta unit
                unit_str, ref_str = m.groups()
                ref = pd.to_datetime(ref_str)
                td_unit = {"seconds": "s", "second": "s", "minutes": "m", "hours": "h", "days": "D"}.get(unit_str.lower(), "s")
                times = ref + pd.to_timedelta(clean, unit=td_unit)
            

            start_ts = times.min()
            end_ts = times.max()

            ping_records.append({
                'id': zarr_key.split('/')[-1].replace(".zarr", ""),
                'ping_start': start_ts,
                'ping_end': end_ts,
                'ping_start_ms': start_ts.value / 1e6,
                'ping_end_ms': end_ts.value / 1e6
            })

    return pd.DataFrame(ping_records).sort_values(by="ping_start").reset_index(drop=True)


years = [2017] # Determined by the region label csv files
ping_time_df = load_mvbs_zarr_s3(zarr_prefix, years)
ping_time_df

Processing year: 2017
Found 104 zarr in: hake_data/data_zarr/MVBS/2017/
174 invalid ping times in: agr230002-bucket01/hake_data/data_zarr/MVBS/2017/x0062_0_wt_20170731_031153_f0003.zarr
2 invalid ping times in: agr230002-bucket01/hake_data/data_zarr/MVBS/2017/x0062_10_wt_20170731_194315_f0003.zarr
732 invalid ping times in: agr230002-bucket01/hake_data/data_zarr/MVBS/2017/x0062_2_wt_20170731_130448_f0004.zarr
312 invalid ping times in: agr230002-bucket01/hake_data/data_zarr/MVBS/2017/x0062_4_wt_20170731_145057_f0004.zarr
299 invalid ping times in: agr230002-bucket01/hake_data/data_zarr/MVBS/2017/x0062_6_wt_20170731_164016_f0003.zarr
15 invalid ping times in: agr230002-bucket01/hake_data/data_zarr/MVBS/2017/x0062_8_wt_20170731_180848_f0002.zarr
1138 invalid ping times in: agr230002-bucket01/hake_data/data_zarr/MVBS/2017/x0063_0_wt_20170731_230214_f0014.zarr


,id,ping_start,ping_end,ping_start_ms,ping_end_ms
0,x0001_0_wt_20170625_165809_f0003,2017-06-25 16:58:10,2017-06-25 17:57:05,1.498410e+12,1.498413e+12
1,x0001_2_wt_20170625_191315_f0003,2017-06-25 19:13:15,2017-06-25 20:11:45,1.498418e+12,1.498422e+12
2,x0001_4_wt_20170625_212354_f0002,2017-06-25 21:23:55,2017-06-25 21:47:55,1.498426e+12,1.498427e+12
3,x0001_6_wt_20170625_225249_f0002,2017-06-25 22:52:50,2017-06-25 23:02:40,1.498431e+12,1.498432e+12
4,x0001_8_wt_20170626_001831_f0003,2017-06-26 00:18:30,2017-06-26 01:10:55,1.498436e+12,1.498439e+12
...,...,...,...,...,...
99,x0062_4_wt_20170731_145057_f0004,2017-07-31 14:50:55,2017-10-25 16:11:07,1.501513e+12,1.508948e+12
100,x0062_6_wt_20170731_164016_f0003,2017-07-31 16:40:15,2017-07-31 17:33:51,1.501519e+12,1.501522e+12
101,x0062_8_wt_20170731_180848_f0002,2017-07-31 18:08:45,2017-07-31 19:17:01,1.501525e+12,1.501529e+12
102,x0062_10_wt_20170731_194315_f0003,2017-07-31 19:43:15,2017-07-31 20:51:31,1.501530e+12,1.501534e+12


In [ ]:
# Local version of the function

'''
def load_mvbs_zarr(mvbs_dir, max_seconds = 1e8): # How to determine 'max_seconds'?
    ping_records = []

    # Loop over each Zarr within that year
    for zarr_path in mvbs_dir.glob("*.zarr"):
        #print(f"Loading {zarr_path}")        
        try:
            ds = xr.open_zarr(str(zarr_path))
            arr = ds['ping_time'].values
            
            if np.issubdtype(arr.dtype, np.datetime64):
                times = pd.to_datetime(arr)
            else:
                raise ValueError("Not datetime64")
        
        # Fallback: raw offsets (seconds) + manual conversion
        except ValueError:
            ds = xr.open_zarr(str(zarr_path), decode_times=False)
            raw = ds['ping_time'].values
                
            # filter out extreme values
            clean = raw[(raw >= 0) & (raw <= max_seconds)]
            if clean.size == 0:
                print(f"Skipping {zarr_path.name}: no valid ping times")
                continue

            # parse “<unit> since <ref>”
            units = ds["ping_time"].attrs.get("units", "")
            m = re.search(r"(\w+)\s+since\s+(.+)", units)
            if not m:
                print(f"Skipping {zarr_path.name}: malformed units “{units}”")
                continue
            
            unit_str, ref_str = m.groups()
            ref = pd.to_datetime(ref_str)

            # map to pandas timedelta unit
            unit_map = {
                "seconds": "s", "second": "s",
                "minutes": "m", "hours": "h", "days": "D"
            }
            td_unit = unit_map.get(unit_str.lower(), "s")
            times = ref + pd.to_timedelta(clean, unit = td_unit)

        start_ts = times.min()
        end_ts = times.max()
        
        ping_records.append({
            'id': zarr_path.stem,                       
            'ping_start': start_ts,  
            'ping_end': end_ts, 
            'ping_start_ms': start_ts.value / 1e6, 
            'ping_end_ms': end_ts.value / 1e6
        })

    ping_time_df = pd.DataFrame(ping_records).sort_values(by="ping_start").reset_index(drop=True)
        
    return ping_time_df


mvbs_dir = Path('mvbs_sample/2017')
ping_time_df_local = load_mvbs_zarr(mvbs_dir)
ping_time_df_local
'''

,id,ping_start,ping_end,ping_start_ms,ping_end_ms
0,x0001_0_wt_20170625_165809_f0003,2017-06-25 16:58:10,2017-06-25 17:57:05,1.498410e+12,1.498413e+12
1,x0001_2_wt_20170625_191315_f0003,2017-06-25 19:13:15,2017-06-25 20:11:45,1.498418e+12,1.498422e+12
2,x0001_4_wt_20170625_212354_f0002,2017-06-25 21:23:55,2017-06-25 21:47:55,1.498426e+12,1.498427e+12
3,x0001_6_wt_20170625_225249_f0002,2017-06-25 22:52:50,2017-06-25 23:02:40,1.498431e+12,1.498432e+12
4,x0001_8_wt_20170626_001831_f0003,2017-06-26 00:18:30,2017-06-26 01:10:55,1.498436e+12,1.498439e+12
...,...,...,...,...,...
99,x0062_4_wt_20170731_145057_f0004,2017-07-31 14:50:55,2017-07-31 17:07:27,1.501513e+12,1.501521e+12
100,x0062_6_wt_20170731_164016_f0003,2017-07-31 16:40:15,2017-07-31 17:16:47,1.501519e+12,1.501521e+12
101,x0062_8_wt_20170731_180848_f0002,2017-07-31 18:08:45,2017-08-01 12:53:13,1.501525e+12,1.501592e+12
102,x0062_10_wt_20170731_194315_f0003,2017-07-31 19:43:15,2017-07-31 20:23:15,1.501530e+12,1.501533e+12


### TODO: Optimize Matching
Check which year exists in region labels first and only load those specific years of zarr to match

1. Create region_time_df
2. Check which year is present in region_time_df
3. Load those specific years of zarr data, create ping_time_df_year (or merge into 1 ping_time_df)
4. Find match

In [ ]:
# Using milliseconds for more precise matching
def find_matching_zarrs(row):
    mask = (
        (row['start_time_ms'] <= ping_time_df['ping_end_ms']) &
        (row['end_time_ms'] >= ping_time_df['ping_start_ms'])
    )
    return ping_time_df.loc[mask, 'id'].tolist()

# Apply it to get a list of Zarr paths per region
region_time_df['matching_zarrs'] = region_time_df.apply(find_matching_zarrs, axis=1)

region_time_df

In [ ]:
# Check matching zarrs
def check_matches_simple(region_time_df, ping_time_df):
    # Show example for first 5 regions
    for i in range(min(5, len(region_time_df))):
        region = region_time_df.iloc[i]
        print(f"\nRegion: {region['id']}")
        print(f"Region time: {region['start_time']} to {region['end_time']}")
        print(f"Matches: {region['matching_zarrs']}")
        
        # Show times for matched MVBS files
        for mvbs_id in region['matching_zarrs']:
            mvbs = ping_time_df[ping_time_df['id'] == mvbs_id].iloc[0]
            print(f"  - {mvbs_id}: {mvbs['ping_start']} to {mvbs['ping_end']}")
        print("-" * 50)

# Check if any regions have no matches
no_matches = region_time_df[region_time_df['matching_zarrs'].apply(len) == 0]
print(f"Regions with no matches: {len(no_matches)} out of {len(region_time_df)}")

# Basic statistics
match_counts = region_time_df['matching_zarrs'].apply(len)
print(f"Average matches per region: {match_counts.mean():.2f}")
print(f"Max matches for a region: {match_counts.max()}")

# Run the check
check_matches_simple(region_time_df, ping_time_df)

# Issues

When loading the data directly from s3, ```ds['ping_time'].values``` are different from data reading from local 

### Abnormal
Example file: ```2017/x0062_8_wt_20170731_180848_f0002.zarr```

Load directly from s3

In [9]:
# Test loading a zarr file
bucket = "agr230002-bucket01"
zarr_key = "hake_data/data_zarr/MVBS/2017/x0062_8_wt_20170731_180848_f0002.zarr"
full_path = f"s3://{bucket}/{zarr_key}"

s3_ds = xr.open_zarr(fs.get_mapper(full_path), decode_times=False)
s3_ds

<xarray.Dataset> Size: 6MB
Dimensions:            (channel: 3, ping_time: 344, depth: 759)
Coordinates:
  * channel            (channel) <U37 444B '' '' ''
  * depth              (depth) float64 6kB nan nan nan nan ... nan nan nan nan
  * ping_time          (ping_time) int64 3kB 0 4714471308 ... 4616075712
Data variables:
    Sv                 (channel, ping_time, depth) float64 6MB dask.array<chunksize=(3, 344, 759), meta=np.ndarray>
    frequency_nominal  (channel) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    latitude           (ping_time) float64 3kB dask.array<chunksize=(344,), meta=np.ndarray>
    longitude          (ping_time) float64 3kB dask.array<chunksize=(344,), meta=np.ndarray>
Attributes:
    processing_function:          commongrid.compute_MVBS
    processing_level:             Level 3A
    processing_level_url:         https://echopype.readthedocs.io/en/stable/p...
    processing_software_name:     echopype
    processing_software_version:  0.9.0
    processing_time:              2024-08-14T00:14:32Z

In [10]:
s3_ds['ping_time'].values

array([                  0,          4714471308,                   0,
                4714471308, 7939626486501289804, 7305790156125334881,
                         0,                  35,                  40,
                        45,                  50,                  55,
                        60,                  65,                  70,
                        75,                  80,                  85,
                        90,                  95,                 100,
                       105,                 110,                 115,
                       120,                 125,                 130,
                       135,                 140,                 145,
                       150,                 155,                 160,
                       165,                 170,                 175,
                       180,                 185,                 190,
                       195,                 200,                 205,
                    

In [11]:
s3_ds['ping_time'].attrs

{'axis': 'T',
 'calendar': 'proleptic_gregorian',
 'long_name': 'Ping time',
 'standard_name': 'time',
 'units': 'seconds since 2017-07-31 18:08:45'}

Copied from s3 to local (have weird ping_time values)

In [12]:
ds = xr.open_zarr('mvbs_sample/2017/x0062_8_wt_20170731_180848_f0002.zarr', decode_times=False)
ds

<xarray.Dataset> Size: 6MB
Dimensions:            (channel: 3, ping_time: 344, depth: 759)
Coordinates:
  * channel            (channel) <U37 444B '' '' ''
  * depth              (depth) float64 6kB nan nan nan nan ... nan nan nan nan
  * ping_time          (ping_time) int64 3kB 0 3782820132520198796 ... 0 0
Data variables:
    Sv                 (channel, ping_time, depth) float64 6MB dask.array<chunksize=(3, 344, 759), meta=np.ndarray>
    frequency_nominal  (channel) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    latitude           (ping_time) float64 3kB dask.array<chunksize=(344,), meta=np.ndarray>
    longitude          (ping_time) float64 3kB dask.array<chunksize=(344,), meta=np.ndarray>
Attributes:
    processing_function:          commongrid.compute_MVBS
    processing_level:             Level 3A
    processing_level_url:         https://echopype.readthedocs.io/en/stable/p...
    processing_software_name:     echopype
    processing_software_version:  0.9.0
    processing_time:              2024-08-14T00:14:32Z

In [13]:
# Values are in seconds since a time reference (ds['ping_time'].attrs['units'])'
# How to determine 'max_seconds'?
# Values differ everytime
ds['ping_time'].values

array([                   0,  3782820132520198796,           5807067136,
        3028958878108044837,   864973762930821254,  3461915200917078021,
         149469828309255689,   217880530152149845,  8603368468337132629,
        3564992126555343392,   505622529647456276,  8747524786399873811,
        3467263160835719968,  1518568513076397596,  5928236109583968340,
        7299807592561594964,  3471766700300400500,  2671482321101850156,
        8319681593119953749,  7296711286213189748,  8386093285481477234,
        7526769799820373865,  1665822210375905903,  3472330516725969421,
         943321809970802736,  3689073928278389555,  3551422945223326005,
         433564935608938507,  1166452306549735955,  5262758573302416134,
        3635642324234892398,   312159074083939889,  7885595613501133571,
        2324230494896418669,  8243086091055548233,  3684016251752047990,
         650221698729214512,    72354425775621674,    40252579577332993,
       -9150890020423237072, -8471026814169993984, 

In [14]:
ds['ping_time'].attrs

{'axis': 'T',
 'calendar': 'proleptic_gregorian',
 'long_name': 'Ping time',
 'standard_name': 'time',
 'units': 'seconds since 2017-07-31 18:08:45'}

### Normal
Example file: ```2017/x0027_0_wt_20170709_214828_f0003.zarr```

Load from s3

In [15]:
# test loading a zarr file

bucket = "agr230002-bucket01"
zarr_key = "hake_data/data_zarr/MVBS/2017/x0027_0_wt_20170709_214828_f0003.zarr"
full_path = f"s3://{bucket}/{zarr_key}"

s3_ds_norm = xr.open_zarr(fs.get_mapper(full_path))
s3_ds_norm

<xarray.Dataset> Size: 14MB
Dimensions:            (channel: 3, ping_time: 748, depth: 759)
Coordinates:
  * channel            (channel) <U37 444B 'GPT  18 kHz 009072058c8d 1-1 ES18...
  * depth              (depth) float64 6kB 0.0 1.0 2.0 3.0 ... 756.0 757.0 758.0
  * ping_time          (ping_time) datetime64[ns] 6kB 2017-07-09T21:48:25 ......
Data variables:
    Sv                 (channel, ping_time, depth) float64 14MB dask.array<chunksize=(3, 748, 759), meta=np.ndarray>
    frequency_nominal  (channel) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    latitude           (ping_time) float64 6kB dask.array<chunksize=(748,), meta=np.ndarray>
    longitude          (ping_time) float64 6kB dask.array<chunksize=(748,), meta=np.ndarray>
Attributes:
    processing_function:          commongrid.compute_MVBS
    processing_level:             Level 3A
    processing_level_url:         https://echopype.readthedocs.io/en/stable/p...
    processing_software_name:     echopype
    processing_software_version:  0.9.0
    processing_time:              2024-08-14T00:26:30Z

In [16]:
s3_ds_norm['ping_time'].values

array(['2017-07-09T21:48:25.000000000', '2017-07-09T21:48:30.000000000',
       '2017-07-09T21:48:35.000000000', '2017-07-09T21:48:40.000000000',
       '2017-07-09T21:48:45.000000000', '2017-07-09T21:48:50.000000000',
       '2017-07-09T21:48:55.000000000', '2017-07-09T21:49:00.000000000',
       '2017-07-09T21:49:05.000000000', '2017-07-09T21:49:10.000000000',
       '2017-07-09T21:49:15.000000000', '2017-07-09T21:49:20.000000000',
       '2017-07-09T21:49:25.000000000', '2017-07-09T21:49:30.000000000',
       '2017-07-09T21:49:35.000000000', '2017-07-09T21:49:40.000000000',
       '2017-07-09T21:49:45.000000000', '2017-07-09T21:49:50.000000000',
       '2017-07-09T21:49:55.000000000', '2017-07-09T21:50:00.000000000',
       '2017-07-09T21:50:05.000000000', '2017-07-09T21:50:10.000000000',
       '2017-07-09T21:50:15.000000000', '2017-07-09T21:50:20.000000000',
       '2017-07-09T21:50:25.000000000', '2017-07-09T21:50:30.000000000',
       '2017-07-09T21:50:35.000000000', '2017-07-09

Load from local

In [17]:
ds_norm = xr.open_zarr('mvbs_sample/2017/x0027_0_wt_20170709_214828_f0003.zarr')
ds_norm

<xarray.Dataset> Size: 14MB
Dimensions:            (channel: 3, ping_time: 748, depth: 759)
Coordinates:
  * channel            (channel) <U37 444B 'GPT  18 kHz 009072058c8d 1-1 ES18...
  * depth              (depth) float64 6kB 0.0 1.0 2.0 3.0 ... 756.0 757.0 758.0
  * ping_time          (ping_time) datetime64[ns] 6kB 2017-07-09T21:48:25 ......
Data variables:
    Sv                 (channel, ping_time, depth) float64 14MB dask.array<chunksize=(3, 748, 759), meta=np.ndarray>
    frequency_nominal  (channel) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    latitude           (ping_time) float64 6kB dask.array<chunksize=(748,), meta=np.ndarray>
    longitude          (ping_time) float64 6kB dask.array<chunksize=(748,), meta=np.ndarray>
Attributes:
    processing_function:          commongrid.compute_MVBS
    processing_level:             Level 3A
    processing_level_url:         https://echopype.readthedocs.io/en/stable/p...
    processing_software_name:     echopype
    processing_software_version:  0.9.0
    processing_time:              2024-08-14T00:26:30Z

In [18]:
ds_norm['ping_time'].values

array(['2017-07-09T21:48:25.000000000', '2017-07-09T21:48:30.000000000',
       '2017-07-09T21:48:35.000000000', '2017-07-09T21:48:40.000000000',
       '2017-07-09T21:48:45.000000000', '2017-07-09T21:48:50.000000000',
       '2017-07-09T21:48:55.000000000', '2017-07-09T21:49:00.000000000',
       '2017-07-09T21:49:05.000000000', '2017-07-09T21:49:10.000000000',
       '2017-07-09T21:49:15.000000000', '2017-07-09T21:49:20.000000000',
       '2017-07-09T21:49:25.000000000', '2017-07-09T21:49:30.000000000',
       '2017-07-09T21:49:35.000000000', '2017-07-09T21:49:40.000000000',
       '2017-07-09T21:49:45.000000000', '2017-07-09T21:49:50.000000000',
       '2017-07-09T21:49:55.000000000', '2017-07-09T21:50:00.000000000',
       '2017-07-09T21:50:05.000000000', '2017-07-09T21:50:10.000000000',
       '2017-07-09T21:50:15.000000000', '2017-07-09T21:50:20.000000000',
       '2017-07-09T21:50:25.000000000', '2017-07-09T21:50:30.000000000',
       '2017-07-09T21:50:35.000000000', '2017-07-09

In [19]:
ds_norm['ping_time'].attrs

{'axis': 'T', 'long_name': 'Ping time', 'standard_name': 'time'}